# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [12]:
! pwd

/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/1-LLM Engineer/Day-4/1-company-brochre


In [13]:
# imports
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from IPython.display import Markdown, display, update_display

In [14]:
# Load env
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
env_path = repo_root / ".env"
load_dotenv(env_path, override=True)
print(f"Loaded env from: {env_path}")


Loaded env from: /home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/.env


## Common LLM call

In [15]:
model = os.getenv("GPT_MODEL", "gpt-4o-mini")

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)


def call_llm(messages, json_mode=False):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"} if json_mode else {"type": "text"},
        timeout=120.0,
    )
    return response.choices[0].message.content


def stream_llm(messages, llm_client=None, llm_model=None):
    llm_client = llm_client or client
    llm_model = llm_model or model
    stream = llm_client.chat.completions.create(
        model=llm_model,
        messages=messages,
        stream=True,
        timeout=120.0,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if not chunk.choices:
            continue
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response


In [16]:
# here is the question; type over this to ask something new

question = """
Explain how the CUDA thread hierarchy (grid → blocks → threads) maps to a GPU's physical hardware (SMs, warps, cores).
"""

## System prompt

A tutor that explains technical questions in markdown.

In [17]:
system_prompt = """
You are a helpful technical tutor who answers questions about python code,
software engineering, data science and LLMs.
Respond in markdown without wrapping the whole answer in a code block.
Be clear, structured and practical.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question},
]

In [18]:
# Get gpt-4o-mini to answer, with streaming
stream_llm(messages)

Here’s the mapping in layers, from the CUDA programming model (what you write) to the GPU hardware (what actually runs):

---

## 1. CUDA programming model: grid → blocks → threads

When you write a kernel:

```cpp
__global__ void myKernel(...) {
    int bx = blockIdx.x;   // block index
    int tx = threadIdx.x;  // thread index within block
    ...
}
```

You conceptually define:

- **Grid**: all threads launched for a kernel call.
- **Block**: a group of threads that:
  - share fast on-chip **shared memory**
  - can synchronize with `__syncthreads()`
- **Thread**: the smallest unit in your code (each thread runs the kernel function with its own indices and registers).

Example: `kernel<<<numBlocks, threadsPerBlock>>>(...)` defines:
- A **grid** of `numBlocks` blocks
- Each **block** has `threadsPerBlock` threads

This is purely the *logical* model. The hardware executes it in a specific way.

---

## 2. Hardware building blocks: SMs, warps, cores

On the GPU, the main relevant hardware units are:

- **Streaming Multiprocessor (SM)**:
  - A “compute unit” that runs blocks.
  - Has:
    - a pool of registers
    - shared memory / L1 cache
    - multiple **cores** (ALUs)
    - instruction issue logic
  - Several SMs exist per GPU (e.g., 80 SMs).

- **Warp**:
  - A group of **32 threads** (on NVIDIA hardware) that execute *in lockstep*.
  - This is the *real hardware scheduling unit*:
    - One instruction is issued per warp per cycle (depending on architecture and occupancy).
  - Threads in a warp share a program counter but can diverge using predication and control flow (at a performance cost).

- **Core** (CUDA core):
  - An execution pipeline that performs integer/FP operations.
  - Several cores per SM (e.g., 64–128 cores per SM depending on architecture).
  - When a warp executes an instruction like `add`, the SM’s scheduler issues that instruction, and the cores execute it for each active thread lane in the warp.

---

## 3. Mapping: grid → blocks → threads onto SMs → warps → cores

### 3.1 Grid → SMs

- A **kernel launch** creates a grid of blocks.
- The **grid is not bound to a single SM**:
  - Blocks from a grid are distributed across all SMs that are available.
  - The runtime/hardware scheduler decides which SM gets which block.

Think of it as:
- Grid = work queue
- SMs = workers
- Runtime = dispatcher that assigns blocks from the queue to workers

### 3.2 Blocks → SMs

- **A block is executed entirely on one SM**:
  - A block never migrates between SMs.
  - All threads in a block share that SM’s:
    - shared memory
    - registers (allocated per thread but from the SM’s register file)
- Multiple blocks can reside on the same SM *concurrently* if resources allow:
  - Limited by:
    - max threads per SM
    - max blocks per SM
    - registers per SM
    - shared memory per SM

This is why you talk about **occupancy**: how many warps/blocks are resident per SM.

### 3.3 Threads → warps

Inside a block:

- The hardware groups threads into **warps** of 32 threads.
- This grouping is deterministic:
  - Thread indices are linearized; e.g., in 1D:
    - Threads 0–31 → warp 0
    - Threads 32–63 → warp 1
    - etc.
- Warps are the real scheduling entities:
  - The SM scheduler selects warps to run instructions.
  - All threads in a warp execute the same instruction at the same time (SIMT: single instruction, multiple threads).

So mapping is:

> Threads (your abstraction) → grouped as warps (hardware execution units)

You don’t explicitly manage warps in CUDA code, but warp behavior is crucial for performance (divergence, memory coalescing, etc.).

### 3.4 Warps → cores

On an SM:

- Several warps are **resident** (ready to run).
- The SM’s **warp scheduler** picks one (or more) ready warps and issues an instruction for them.
- For a given issued instruction:
  - Each lane of the warp (each thread) gets its work done by the SM’s **cores**.
  - If an SM has, say, 64 cores and the warp has 32 active threads:
    - It can execute that instruction for all 32 threads in a single cycle (in a simplified view).
- Different types of instructions use different pipelines:
  - FP units, integer units, load/store units, tensor cores, etc.

The warp is thus mapped to the SM’s cores:
- Warps are scheduled
- Cores perform the actual arithmetic / memory ops for the warp’s threads.

---

## 4. Putting it all together with an example

Suppose you launch:

```cpp
dim3 gridDim(80);       // 80 blocks
dim3 blockDim(256);     // 256 threads per block
myKernel<<<gridDim, blockDim>>>();
```

Total threads: 80 × 256 = 20,480.

Hardware mapping:

1. **Grid**:
   - 80 blocks form the grid.

2. **Blocks → SMs**:
   - Assume the GPU has 20 SMs.
   - The runtime may distribute blocks roughly evenly:
     - Each SM gets blocks as it becomes available.
     - E.g., initially 4 blocks per SM if resources allow.

3. **Threads → warps**:
   - Each block: 256 threads → 8 warps (256 / 32 = 8).
   - So per SM (with 4 blocks resident): 4 × 8 = 32 warps resident.

4. **Warps → cores**:
   - The SM scheduler chooses warps to execute each cycle.
   - Each chosen warp issues its next instruction.
   - The SM’s cores execute that instruction for the warp’s 32 threads.
   - If some threads are inactive due to divergence, those lanes are masked off.

---

## 5. Key conceptual points and performance implications

- **Blocks-to-SM is static during execution**:
  - Once a block starts on an SM, it stays there.
  - That’s why dynamic parallelism, block size, and resource usage critically affect occupancy and performance.

- **Warps are the real “threads” for performance**:
  - Branch divergence is per warp.
  - Memory coalescing is per warp.
  - Latency hiding: scheduler switches between warps to cover memory latency.

- **Threads are a logical abstraction**:
  - You think in terms of threads for indexing and per-element work.
  - The hardware, however, batches them into warps and runs warps on SM cores.

- **Shared memory and synchronization are block-scoped because blocks map to SMs**:
  - `__syncthreads()` only works within a block (i.e., among warps of that block on one SM).
  - No built-in sync across blocks without special mechanisms (cooperative groups, multiple kernel launches, etc.).

---

## 6. Quick mapping summary

- **Grid** → distributed over all **SMs** (blocks are assigned to SMs)
- **Block** → runs on exactly one **SM**
- **Threads in a block** → grouped into **warps** of 32 threads
- **Warps** → scheduled and executed by an SM’s **cores**

When you’re tuning performance, you’re mainly:
- Choosing block sizes and resource usage to maximize **warps per SM** (occupancy).
- Structuring control flow and memory access patterns to keep **warps efficient** (avoid divergence, improve coalescing).

'Here’s the mapping in layers, from the CUDA programming model (what you write) to the GPU hardware (what actually runs):\n\n---\n\n## 1. CUDA programming model: grid → blocks → threads\n\nWhen you write a kernel:\n\n```cpp\n__global__ void myKernel(...) {\n    int bx = blockIdx.x;   // block index\n    int tx = threadIdx.x;  // thread index within block\n    ...\n}\n```\n\nYou conceptually define:\n\n- **Grid**: all threads launched for a kernel call.\n- **Block**: a group of threads that:\n  - share fast on-chip **shared memory**\n  - can synchronize with `__syncthreads()`\n- **Thread**: the smallest unit in your code (each thread runs the kernel function with its own indices and registers).\n\nExample: `kernel<<<numBlocks, threadsPerBlock>>>(...)` defines:\n- A **grid** of `numBlocks` blocks\n- Each **block** has `threadsPerBlock` threads\n\nThis is purely the *logical* model. The hardware executes it in a specific way.\n\n---\n\n## 2. Hardware building blocks: SMs, warps, cores\n\n